# Customer Segmentation and Purchasing Behaviour Analysis Using Unsupervised Machine Learning

This notebook documents the complete analytical workflow for the UCI Online Retail dataset.

**Workflow**

Raw transactions → Data inspection → Data cleaning → Customer-level feature engineering → Log transformation and standardisation → PCA → K-Means → Agglomerative Clustering → DBSCAN → Model comparison → Final K-Means segmentation → Customer profiling → Visualisation.

The analysis is unsupervised: there is no target variable and no labelled outcome is used during clustering.


## 1. Imports and project setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
from sklearn.neighbors import NearestNeighbors

pd.set_option("display.max_columns", None)

FIGURE_DIR = Path("outputs/figures")
TABLE_DIR = Path("outputs/tables")

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load the UCI Online Retail dataset

In [ ]:
INPUT_FILE = "data/raw/retail.xlsx"

df = pd.read_excel(INPUT_FILE)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()


## 3. Initial data inspection

The dataset contains transaction-level records. The inspection focuses on structure, data types, missing values, duplicate rows, and the numerical range of the transaction variables.


In [ ]:
print("--- DATA TYPES ---")
print(df.dtypes)

print("\n--- MISSING VALUES ---")
print(df.isnull().sum())

print("\n--- DUPLICATE ROWS ---")
print(df.duplicated().sum())

print("\n--- DATE RANGE ---")
print(df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

print("\n--- UNIQUE VALUES ---")
print(df.nunique())

print("\n--- NUMERICAL SUMMARY ---")
print(df.describe())


## 4. Data-quality diagnostics

In [ ]:
diagnostics = {
    "Missing Customer IDs": df["CustomerID"].isna().sum(),
    "Cancellation invoices": df["InvoiceNo"].astype(str).str.startswith("C").sum(),
    "Negative quantities": (df["Quantity"] < 0).sum(),
    "Zero/negative prices": (df["UnitPrice"] <= 0).sum(),
    "Missing descriptions": df["Description"].isna().sum(),
    "Duplicate rows": df.duplicated().sum()
}

pd.Series(diagnostics, name="Count")


## 5. Transaction-level cleaning

The cleaning rules retain transactions that can be reliably linked to customers and represent positive-value purchases.

- Exact duplicate rows are removed.
- Records without `CustomerID` are removed because customer-level segmentation requires a customer identifier.
- Cancellation invoices are removed.
- Non-positive quantities are removed.
- Non-positive unit prices are removed.
- `Description` is not required for the customer-level feature set.
- `TotalAmount` is calculated as quantity multiplied by unit price.


In [ ]:
clean = df.copy()

clean = clean.drop_duplicates()
clean = clean.dropna(subset=["CustomerID"])
clean = clean[~clean["InvoiceNo"].astype(str).str.startswith("C")]
clean = clean[clean["Quantity"] > 0]
clean = clean[clean["UnitPrice"] > 0]

clean["TotalAmount"] = clean["Quantity"] * clean["UnitPrice"]

print("Original shape:", df.shape)
print("Cleaned shape:", clean.shape)
print("Unique customers:", clean["CustomerID"].nunique())
print("Date range:", clean["InvoiceDate"].min(), "to", clean["InvoiceDate"].max())
print("Total sales value:", round(clean["TotalAmount"].sum(), 2))

print("\nMissing values after cleaning:")
print(clean.isnull().sum())


## 6. Save cleaned transaction data

In [ ]:
clean.to_csv(
    "data/processed/clean_retail.csv",
    index=False
)


## 7. Customer-level feature engineering

Customer behaviour is represented using RFM-style measures together with transaction-volume and product-diversity features.

- **Recency:** days since the customer's most recent purchase.
- **Frequency:** number of distinct invoices.
- **Monetary:** total transaction value.
- **TotalQuantity:** total number of items purchased.
- **ProductDiversity:** number of distinct stock codes purchased.
- **AverageOrderValue:** mean value of a customer's orders.

The reference date is one day after the final transaction in the cleaned dataset.


In [ ]:
reference_date = clean["InvoiceDate"].max() + pd.Timedelta(days=1)

customer_features = (
    clean.groupby("CustomerID")
    .agg(
        Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("TotalAmount", "sum"),
        TotalQuantity=("Quantity", "sum"),
        ProductDiversity=("StockCode", "nunique")
    )
    .reset_index()
)

order_values = (
    clean.groupby(["CustomerID", "InvoiceNo"])["TotalAmount"]
    .sum()
    .groupby("CustomerID")
    .mean()
    .rename("AverageOrderValue")
)

customer_features = customer_features.merge(
    order_values,
    on="CustomerID",
    how="left"
)

print("Customer feature shape:", customer_features.shape)
customer_features.head()


In [ ]:
customer_features.describe()


## 8. Prepare features for clustering

The customer-level variables are strongly right-skewed. Log transformation is applied using `log1p` to reduce the influence of extreme values. The transformed features are then standardised so that variables with larger numerical scales do not dominate the clustering process.


In [ ]:
feature_columns = [
    "Recency",
    "Frequency",
    "Monetary",
    "TotalQuantity",
    "ProductDiversity",
    "AverageOrderValue"
]

X = customer_features[feature_columns].copy()

for column in feature_columns:
    X[column] = np.log1p(X[column])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=feature_columns
)

print("Shape:", X_scaled_df.shape)
print("\nMeans:")
print(X_scaled_df.mean().round(4))

print("\nStandard deviations:")
print(X_scaled_df.std().round(4))


In [ ]:
X_scaled_df.to_csv(
    "data/processed/clustering_features.csv",
    index=False
)


## 9. PCA for dimensionality reduction and visualisation

PCA is used to project the six-dimensional customer feature space into two dimensions for visual inspection. PCA is not used as the clustering input; clustering is performed on all six standardised features.


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_

print("PC1:", round(explained[0] * 100, 2), "%")
print("PC2:", round(explained[1] * 100, 2), "%")
print("Combined:", round(explained[:2].sum() * 100, 2), "%")


In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    s=18,
    alpha=0.5
)

plt.xlabel(f"PC1 ({explained[0] * 100:.2f}% variance)")
plt.ylabel(f"PC2 ({explained[1] * 100:.2f}% variance)")
plt.title("Customer Feature Space Using PCA")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "pca_customer_space.png", dpi=300)
plt.show()


## 10. K-Means model selection

K-Means is evaluated for values of K from 2 to 10. Three internal validation measures are considered:

- Silhouette Score — higher is better.
- Davies-Bouldin Index — lower is better.
- Calinski-Harabasz Index — higher is better.


In [ ]:
kmeans_results = []

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_scaled)

    kmeans_results.append({
        "K": k,
        "Silhouette": silhouette_score(X_scaled, labels),
        "Davies_Bouldin": davies_bouldin_score(X_scaled, labels),
        "Calinski_Harabasz": calinski_harabasz_score(X_scaled, labels)
    })

kmeans_selection = pd.DataFrame(kmeans_results)

print(kmeans_selection.round(4).to_string(index=False))

kmeans_selection.to_csv(
    TABLE_DIR / "kmeans_selection.csv",
    index=False
)


### K-Means selection result

All three internal validation measures identify **K=2** as the strongest K-Means solution in the tested range.


## 11. Agglomerative Clustering model selection

Ward-linkage Agglomerative Clustering is evaluated for two through ten clusters using the same three internal validation measures.


In [ ]:
agg_results = []

for k in range(2, 11):
    model = AgglomerativeClustering(
        n_clusters=k,
        linkage="ward"
    )

    labels = model.fit_predict(X_scaled)

    agg_results.append({
        "K": k,
        "Silhouette": silhouette_score(X_scaled, labels),
        "Davies_Bouldin": davies_bouldin_score(X_scaled, labels),
        "Calinski_Harabasz": calinski_harabasz_score(X_scaled, labels)
    })

agg_selection = pd.DataFrame(agg_results)

print(agg_selection.round(4).to_string(index=False))

agg_selection.to_csv(
    TABLE_DIR / "agglomerative_selection.csv",
    index=False
)


## 12. DBSCAN parameter investigation

A five-neighbour k-distance curve is used to examine the distance structure of the standardised customer feature space. DBSCAN is then evaluated over a range of epsilon values with `min_samples=5`.


In [ ]:
neighbors = NearestNeighbors(n_neighbors=5)
neighbors.fit(X_scaled)

distances, _ = neighbors.kneighbors(X_scaled)

k_distances = np.sort(distances[:, -1])

print("K-distance summary:")
print("Minimum:", round(k_distances.min(), 4))
print("25th percentile:", round(np.percentile(k_distances, 25), 4))
print("Median:", round(np.median(k_distances), 4))
print("75th percentile:", round(np.percentile(k_distances, 75), 4))
print("90th percentile:", round(np.percentile(k_distances, 90), 4))
print("95th percentile:", round(np.percentile(k_distances, 95), 4))
print("Maximum:", round(k_distances.max(), 4))


In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(k_distances)
plt.xlabel("Sorted observations")
plt.ylabel("5-nearest-neighbour distance")
plt.title("DBSCAN k-Distance Curve")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "dbscan_k_distance.png", dpi=300)
plt.show()


In [ ]:
eps_values = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]

dbscan_results = []

for eps in eps_values:

    model = DBSCAN(
        eps=eps,
        min_samples=5
    )

    labels = model.fit_predict(X_scaled)

    valid_mask = labels != -1
    labels_valid = labels[valid_mask]
    X_valid = X_scaled[valid_mask]

    n_clusters = len(set(labels_valid))

    if n_clusters >= 2:
        silhouette = silhouette_score(X_valid, labels_valid)
        davies_bouldin = davies_bouldin_score(X_valid, labels_valid)
        calinski_harabasz = calinski_harabasz_score(X_valid, labels_valid)
    else:
        silhouette = np.nan
        davies_bouldin = np.nan
        calinski_harabasz = np.nan

    dbscan_results.append({
        "eps": eps,
        "clusters": n_clusters,
        "noise_count": int((labels == -1).sum()),
        "noise_percentage": (labels == -1).mean() * 100,
        "silhouette": silhouette,
        "davies_bouldin": davies_bouldin,
        "calinski_harabasz": calinski_harabasz
    })

dbscan_selection = pd.DataFrame(dbscan_results)

print(dbscan_selection.round(4).to_string(index=False))

dbscan_selection.to_csv(
    TABLE_DIR / "dbscan_evaluation.csv",
    index=False
)


## 13. Final model comparison

The selected configurations are:

- K-Means: `n_clusters=2`, `random_state=42`, `n_init=20`
- Agglomerative: `n_clusters=2`, `linkage="ward"`
- DBSCAN: `eps=1.1`, `min_samples=5`

For DBSCAN, noise observations are excluded from internal metric calculations because they are not assigned to a cluster.


In [ ]:
models = {
    "KMeans": KMeans(
        n_clusters=2,
        random_state=42,
        n_init=20
    ),
    "Agglomerative": AgglomerativeClustering(
        n_clusters=2,
        linkage="ward"
    ),
    "DBSCAN": DBSCAN(
        eps=1.1,
        min_samples=5
    )
}

results = []
cluster_profiles = {}

for name, model in models.items():

    labels = model.fit_predict(X_scaled)

    valid_mask = labels != -1
    X_valid = X_scaled[valid_mask]
    labels_valid = labels[valid_mask]

    n_clusters = len(set(labels_valid))

    if n_clusters >= 2:
        silhouette = silhouette_score(X_valid, labels_valid)
        davies_bouldin = davies_bouldin_score(X_valid, labels_valid)
        calinski_harabasz = calinski_harabasz_score(X_valid, labels_valid)
    else:
        silhouette = np.nan
        davies_bouldin = np.nan
        calinski_harabasz = np.nan

    results.append({
        "Model": name,
        "Clusters": n_clusters,
        "Noise": int((labels == -1).sum()),
        "Silhouette": silhouette,
        "Davies_Bouldin": davies_bouldin,
        "Calinski_Harabasz": calinski_harabasz
    })

    model_customers = customer_features.copy()
    model_customers["Cluster"] = labels

    profile = (
        model_customers[model_customers["Cluster"] != -1]
        .groupby("Cluster")
        .agg(
            Customers=("CustomerID", "count"),
            Recency=("Recency", "mean"),
            Frequency=("Frequency", "mean"),
            Monetary=("Monetary", "mean"),
            TotalQuantity=("TotalQuantity", "mean"),
            ProductDiversity=("ProductDiversity", "mean"),
            AverageOrderValue=("AverageOrderValue", "mean")
        )
        .reset_index()
    )

    cluster_profiles[name] = profile

    model_customers.to_csv(
        TABLE_DIR / f"{name.lower()}_customer_clusters.csv",
        index=False
    )

results_df = pd.DataFrame(results)

print(results_df.round(4).to_string(index=False))

results_df.to_csv(
    TABLE_DIR / "final_model_comparison.csv",
    index=False
)


## 14. Final model selection

K-Means is selected as the final segmentation method.

The decision is not based on one metric alone. K-Means has the strongest overall performance among the partitioning approaches, with:

- Silhouette = **0.3563**
- Davies-Bouldin = **1.0550**
- Calinski-Harabasz = **3183.4055**

DBSCAN produced stronger Silhouette and Davies-Bouldin values, but its solution was not useful for customer segmentation: 4,257 customers were placed in one cluster, only three customers formed the second cluster, and 78 observations were labelled as noise.

Agglomerative clustering produced interpretable groups but was weaker than K-Means on all three internal validation measures.

Therefore, K-Means with two clusters provides the most useful balance between quantitative performance and business interpretability.


## 15. Final K-Means customer profiles

In [ ]:
final_kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

final_labels = final_kmeans.fit_predict(X_scaled)

final_customers = customer_features.copy()
final_customers["Cluster"] = final_labels

final_profiles = (
    final_customers
    .groupby("Cluster")
    .agg(
        Customers=("CustomerID", "count"),
        Recency=("Recency", "mean"),
        Frequency=("Frequency", "mean"),
        Monetary=("Monetary", "mean"),
        TotalQuantity=("TotalQuantity", "mean"),
        ProductDiversity=("ProductDiversity", "mean"),
        AverageOrderValue=("AverageOrderValue", "mean")
    )
    .reset_index()
)

print(final_profiles.round(2).to_string(index=False))

final_customers.to_csv(
    TABLE_DIR / "kmeans_customer_clusters.csv",
    index=False
)


### Segment interpretation

**Cluster 0 — High-Value Active Customers**

These customers have substantially lower recency, higher purchase frequency, higher monetary contribution, greater product diversity, higher quantities purchased, and higher average order values.

**Cluster 1 — Lower-Value Less-Active Customers**

These customers have higher recency and considerably lower frequency, monetary contribution, product diversity, quantity purchased, and average order value.

These labels describe the observed profiles; they do not imply that customers will permanently remain in the same behavioural group.


## 16. Final PCA visualisation

In [ ]:
plt.figure(figsize=(9, 6))

for cluster in sorted(final_customers["Cluster"].unique()):

    mask = final_customers["Cluster"] == cluster

    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        s=18,
        alpha=0.55,
        label=f"Cluster {cluster}"
    )

plt.xlabel(f"PC1 ({explained[0] * 100:.2f}% variance)")
plt.ylabel(f"PC2 ({explained[1] * 100:.2f}% variance)")
plt.title("Final K-Means Customer Segmentation")
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "final_kmeans_pca_clusters.png",
    dpi=300
)

plt.show()


## 17. Cluster size distribution

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    final_profiles["Cluster"].astype(str),
    final_profiles["Customers"]
)

plt.xlabel("K-Means Cluster")
plt.ylabel("Number of Customers")
plt.title("Customer Distribution Across Final K-Means Clusters")

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "kmeans_cluster_sizes.png",
    dpi=300
)

plt.show()


## 18. Customer profile comparison

In [ ]:
profile_metrics = [
    "Recency",
    "Frequency",
    "Monetary",
    "TotalQuantity",
    "ProductDiversity",
    "AverageOrderValue"
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, metric in zip(axes, profile_metrics):

    ax.bar(
        final_profiles["Cluster"].astype(str),
        final_profiles[metric]
    )

    ax.set_title(metric)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Mean value")

plt.suptitle(
    "K-Means Customer Profile Comparison",
    fontsize=16
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "kmeans_customer_profile_comparison.png",
    dpi=300
)

plt.show()


## 19. Standardised profile comparison

In [ ]:
profile_values = final_profiles[profile_metrics]

profile_z = (
    profile_values - profile_values.mean()
) / profile_values.std()

plt.figure(figsize=(10, 4.5))

plt.imshow(
    profile_z.values,
    aspect="auto"
)

plt.xticks(
    range(len(profile_metrics)),
    profile_metrics,
    rotation=30,
    ha="right"
)

plt.yticks(
    range(len(final_profiles)),
    [f"Cluster {c}" for c in final_profiles["Cluster"]]
)

plt.colorbar(label="Standardized mean")

plt.title("Relative Customer Profile by K-Means Cluster")

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "kmeans_profile_heatmap.png",
    dpi=300
)

plt.show()


## 20. Business interpretation

The segmentation separates customers primarily according to purchasing recency, frequency, monetary contribution, transaction volume, product diversity, and average order value.

The high-value active segment represents customers with stronger historical purchasing activity and therefore provides a useful population for retention and relationship-building initiatives. The lower-value less-active segment provides a different opportunity: the business can investigate ways to encourage repeat purchasing and increase engagement.

The clustering should be treated as a decision-support tool rather than a permanent customer classification. Segment membership can change as additional transaction data become available.


## 21. Limitations

1. The analysis is based on historical transactions from the available observation period and therefore reflects past purchasing behaviour.
2. Customer-level features depend on the chosen aggregation definitions and can produce different segment structures if alternative measures are used.
3. Log transformation and standardisation improve comparability but also influence the resulting cluster geometry.
4. The two-dimensional PCA plots are visual summaries of a six-dimensional clustering space and should not be interpreted as the complete clustering structure.
5. DBSCAN metrics were calculated after excluding noise observations, so its validation scores are not perfectly comparable with the scores from K-Means and Agglomerative Clustering.
6. The analysis does not include demographic, marketing-response, profitability, or customer-service information, so the segments describe purchasing behaviour rather than complete customer value.


## 22. Future work and recommended actions

1. Re-run the segmentation periodically as new transactions become available.
2. Test alternative behavioural features, including purchase intervals and category-level spending.
3. Compare clustering stability across different sampling periods.
4. Investigate additional clustering methods and parameter settings.
5. Link the resulting segments to campaign response or retention outcomes when labelled business data become available.
6. Evaluate whether segment-specific marketing actions produce measurable improvements in repeat purchasing and customer value.


## 23. Reproducibility

The main analysis scripts are stored under `src/`, generated tables under `outputs/tables/`, and figures under `outputs/figures/`.

The notebook is intended to provide a single end-to-end view of the analytical process and can be run from the project root after the required Python packages are installed.
